In [ ]:
# 📦 Step 1: Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# 📂 Step 2: Define Dataset Paths
train_dataset = "c:/Users/admin/Desktop/8980/DL/Data/train"
test_dataset = "c:/Users/admin/Desktop/8980/DL/Data/test"


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import os

def augment_and_save_images(input_dir, output_dir, augment_count_per_image):
    os.makedirs(output_dir, exist_ok=True)

    datagen = ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    all_files = os.listdir(input_dir)
    for i, file in enumerate(all_files):
        img_path = os.path.join(input_dir, file)
        try:
            img = load_img(img_path, target_size=(256, 256))
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)

            j = 0
            for batch in datagen.flow(x, batch_size=1,
                                      save_to_dir=output_dir,
                                      save_prefix=f'aug_{i}',
                                      save_format='jpeg'):
                j += 1
                if j >= augment_count_per_image:
                    break
        except Exception as e:
            print(f"Skipping {file}: {e}")


In [5]:
# COVID19: Need ~2958, have 460 → ~6 per image
augment_and_save_images(
    input_dir="c:/Users/admin/Desktop/8980/DL/Data/train/COVID19",
    output_dir="c:/Users/admin/Desktop/8980/DL/augmented/train/COVID19",
    augment_count_per_image=6
)

# train_dataset = "c:/Users/admin/Desktop/8980/DL/Data/train/COVID19"
# test_dataset = "c:/Users/admin/Desktop/8980/DL/Data/test"
# c:\Users\admin\Desktop\8980\DL\Data\train\COVID19

# NORMAL: Need ~2152, have 1266 → ~2 per image
augment_and_save_images(
    input_dir="c:/Users/admin/Desktop/8980/DL/Data/train/NORMAL",
    output_dir="c:/Users/admin/Desktop/8980/DL/augmented/train/NORMAL",
    augment_count_per_image=2
)


In [ ]:
train_dataset = "c:/Users/admin/Desktop/8980/DL/augmented/train"
test_dataset = "c:/Users/admin/Desktop/8980/DL/Data/test"


In [ ]:
# 🔄 Step 3: Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)


In [ ]:
# 📊 Step 4: Load Data from Directory
train_generator = train_datagen.flow_from_directory(
    train_dataset,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    test_dataset,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical',
    shuffle=False  # for accurate predictions order
)


In [ ]:
# # ⚖️ Step 5: Handle Class Imbalance Using Class Weights
# labels = train_generator.classes
# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(labels),
#     y=labels
# )
# class_weights_dict = dict(enumerate(class_weights))
# print("Class Weights:", class_weights_dict)


In [ ]:
# 🧠 Step 6: Build CNN Model
model = Sequential()

model.add(Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(256, 256, 3), kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(3, activation='softmax'))  # 3 classes

model.summary()


In [ ]:
# 🧠 Step 6: Build CNN Model
model = Sequential()

model.add(Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(256, 256, 3), kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(3, activation='softmax'))  # 3 classes

model.summary()


In [ ]:
# ⚙️ Step 7: Compile Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])



In [ ]:
# 🏋️ Step 8: Train the Model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    # class_weight=class_weights_dict
)


In [ ]:
# 📈 Step 9: Evaluate the Model
test_loss, test_accuracy = model.evaluate(val_generator)
print(f"✅ Test Accuracy: {test_accuracy:.4f}")
print(f"❌ Test Loss: {test_loss:.4f}")


In [ ]:
# 🧪 Step 10: Classification Report & Confusion Matrix
y_pred = model.predict(val_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = val_generator.classes
class_labels = list(val_generator.class_indices.keys())

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred_classes, target_names=class_labels))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# 💾 Step 11: Save the Model
model.save('covid_classifier_balanced.h5')
